# Humility-Enhanced LLM ECE 5424 Robert Corey, Ryan Brown, Yifu Yang

In [ ]:
import os, sys
from pathlib import Path

PROJECT_DIR = '/content/ECE5424Project-main'
if not Path(PROJECT_DIR).exists():
    from google.colab import files
    print('Upload ECE5424Project-main.zip when prompted.')
    uploaded = files.upload()
    [zip_name] = list(uploaded.keys())
    !unzip -q -o "$zip_name" -d /content/

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print('cwd:', os.getcwd())
!ls

In [ ]:
!pip install -q -r requirements.txt
!pip install -q --upgrade transformers accelerate
!pip install -q gradio>=4.0
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '| device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
from google.colab import userdata
tok = userdata.get('HFToken') #Use your token here
os.environ['HF_TOKEN'] = tok
print('Token loaded:', 'yes' if tok else 'no')

In [ ]:
MODEL_NAME = 'meta-llama/Meta-Llama-3.1-8B-Instruct'
BATCH_SIZE = 32
DTYPE = 'bfloat16'

!python scripts/run_baseline.py --model {MODEL_NAME} --batch-size {BATCH_SIZE} --dtype {DTYPE}
!python scripts/run_calibration.py --tag truthfulqa

In [ ]:
import json, pathlib
print(json.dumps(json.loads(pathlib.Path('results/truthfulqa_report.json').read_text()), indent=2))
from IPython.display import Image, display
for kind in ('raw', 'temperature', 'isotonic'):
    display(Image(f'results/truthfulqa_reliability_{kind}.png'))

In [ ]:
os.environ['HF_TOKEN'] = tok
!python scripts/run_freeform_calibration.py \
    --model {MODEL_NAME} --dtype {DTYPE} --max-new-tokens 96

In [ ]:
for kind in ('raw', 'calibrated'):
    display(Image(f'results/freeform_reliability_{kind}.png'))

In [ ]:
os.environ['HF_TOKEN'] = tok
!python scripts/run_chat.py --model {MODEL_NAME} --dtype {DTYPE} --share

In [ ]:
!cd results && zip -qr /content/ece5424_results.zip . && ls -lh /content/ece5424_results.zip
# from google.colab import files; files.download('/content/ece5424_results.zip')